In [ ]:
pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.8 MB/s eta 0:00:00
  Created w

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Batch, SamplerV2 as Sampler, EstimatorV2 as Estimator
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.random import random_circuit

## Objective 1: Execute on hardware



Executing quantum circuits on real hardware involves several steps:

1. **Authentication**: Connecting to the IBM Quantum Runtime service.
2. **Backend Selection**: Choosing an available quantum processor.
3. **Circuit Preparation**: Transpiling circuits for specific hardware constraints.
4. **Job Submission**: Sending circuits for execution..
5. **Result Retrieval**: Obtain and analyze measurement outcomes.

## Objective 2: Execution modes

### Three Execution Modes

IBM Quantum provides three distinct execution modes for running quantum programs:

1. **Job Mode** (`mode=backend`)
   - Single request of Sampler or Estimator without a context manager.
   - Simplest mode for one-off executions.
   - Specify `mode=backend` when initializing the primitive.

2. **Batch Mode** (`mode=batch`)
   - Multi-job workload made up of independently executable jobs, no condoitional relations to each other.
   - Jobs are scheduled together for efficiency.
   - Not gauranteed to run in order of submission, may run in parallel to other users jobs , QPU calibriation jobs may also run between batched jobs.
   - Queuing time doesn't decrease for the first job submitted in a job.
   - Specify `mode=batch` when initializing the primitive.

3. **Session Mode** (`mode=session`)
   - Dedicated window to run a multi-job workload.
   - execlusive access to the QPU and no other jobs can run including calibriation jobs.
   - Particularly useful for iterative tasks that require frequent communication between classical and quantum resources.
   - Queuing time doesn't decrease for the first job submitted in a job.
   - Specify `mode=session` when initializing the primitive.

## Objective 3: Choose execution mode

### When to Use Each Mode

| Mode | Best For | Key Features |
|------|----------|-------------|
| **Batch** | Large collections of independent circuits | • Entire batch scheduled together<br>• Classical computations run in parallel<br>• Minimal delay between jobs<br>• Flexible - cancel remaining jobs if needed<br>• Less expensive than sessions |
| **Session** | Variational algorithms, iterative workloads | • All batch mode functionality<br>• Dedicated access to QPU<br>• Useful when inputs arrive gradually<br>• Better for workloads requiring low latency |
| **Job** | Small experiments, quick tests | • Simplest interface<br>• Might run sooner than batch<br>• Good for learning and testing |

### Best Practices

- Use batch mode by default, especially when all inputs are ready, it is more cost-effective than sessions.

- Batch mode is ideal for submitting multiple primitive jobs at once to reduce total runtime.

- Session mode is best for iterative workflows or when dedicated QPU access is required, but it is generally more expensive.

- Job mode should be used only for a single primitive request.

- Open Plan users cannot use sessions.

### Recommendations:

- Each job has fixed overhead. If a job uses less than 1 minute of QPU time, combine multiple tasks into a single job.

- If jobs take more than 1 minute or cannot be combined, submit multiple jobs in batch or session mode to make use of parallel classical processing (up to 5 jobs), even though the QPU runs one job at a time.

- Tune workload size and parallelism specially in sessions to minimize wall-clock time and cost; splitting large jobs can be more efficient.

## Objective 4 : Run Jobs in Batch

### Batch Execution Fundamentals

Batch mode allows you to submit multiple quantum jobs that are scheduled together. This is more efficient than submitting individual jobs separately.

In [ ]:
# if your account is already saved, this will load it
service = QiskitRuntimeService()
# replace <instance_name> with your instance name
# replace <token> with your IBM Quantum API token
# Uncomment and use the following line if you need to specify instance and token
# srvice = QiskitRuntimeService(channel="ibm_cloud",instance="<instance_name>",token=<token>)


### Open a batch

Can be done by initializing the `Batch` class or by using Context Manager `with Batch()`

In [ ]:
# Method 1: Explicit batch creation and closure
backend = service.least_busy(operational=True, simulator=False)

#Create a batch execution context
batch = Batch(backend=backend, max_time="5m")

# configure primitives to use batch mode
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)

# Close the batch explicitly (no context manager used)
batch.close()

In [ ]:
# Method 2: Using context manager
backend = service.least_busy(operational=True, simulator=False)

print(backend.name)

with Batch(backend=backend, max_time="5m"):
  estimator = Estimator()
  sampler = Sampler()

#Batch automatically closes when context exits

**Prepare circuits to run in batches**

In [ ]:
from qiskit.primitives.containers import estimator_pub
# Create example circuits and observables
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.cx(0, 1)


qc2 = QuantumCircuit(2)
qc2.h(0)
qc2.cx(0, 1)
qc2.rz(0.5, 1)
qc2.measure_all()

# Create observables for estimator
observable1 = SparsePauliOp("Z"*133)
observable2 = SparsePauliOp("X"*133)

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit1 = pm.run(qc1)
isa_circuit2 = pm.run(qc2)

# Create primitive units (pubs) for execution
estimator_pub = (isa_circuit1, observable1)
sampler_pub = isa_circuit2

#### Batch Duration Limits

max TTL can be defined with `max_time` , it starts when the job batch starts, when max time is reached , the batch is closed and any job running will finish but all queued jobs will fail.

| Plan Type | Maximum Batch Duration |
|-----------|-----------------------|
| Open Plan | 10 minutes |
| Paid Plans | 8 hours |

Additionlly there is also interactive TTL that can not be configured (1 minute for all plans), if there is no jobs on the queue within the window, the job is temporarliy deactivated.

**Note:** These limits apply to the total execution time of all jobs within a batch.

### Close a batch

A batch automatically closes when its context manager exits. After that the batch stops accepting new jobs and finishes running all queued or active jobs until the maximum TTL is reached, then which it closes permanently.

If you are not using a context manager, you must close the batch manually. Leaving a batch open and submitting jobs later risks hitting the maximum TTL before they start, causing cancellation. Calling `batch.close()` prevents new submissions while allowing already submitted jobs to finish and their results to be retrieved.

In [ ]:
with Batch(backend=backend) as batch:
  estimator = Estimator()
  sampler = Sampler()
  job1 = estimator.run([estimator_pub])
  job2 = sampler.run([sampler_pub])


# The batch is no longer accepting jobs but the submitted job will run to completion.

result = job1.result()
result2 = job2.result()

<span style="color:red"> **Running the cell below will use around 10-15 seconds from your IBM account credit** </span>

In [ ]:
batch = Batch(backend=backend)

estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])


# Manually close the batch. Running and queued jobs will run to completion.
batch.close()

In [ ]:
# This will raise an error since the batch is closed
job3 = sampler.run([sampler_pub])

### Practical Example: Running Circuits in Batch

<span style="color:red"> **Running the cell below will use around 10-15 seconds from your IBM account credit** </span>

In [ ]:
# Execute circuits using batch mode with context manager
with Batch(backend=backend) as batch:
  estimator = Estimator()
  sampler = Sampler()

  # submit jobs to batch
  estimator_job = estimator.run([estimator_pub])
  sampler_job = sampler.run([sampler_pub])

# Results are available after batch context closes
# The batch is no longer accepting jobs but submitted jobs run to completion
estimator_result = estimator_job.result()
sampler_result = sampler_job.result()

print(estimator_result)
print(sampler_result)

In [ ]:
### Batch Information and Monitoring

with Batch(backend=backend) as batch:
  details = batch.details()

print("Batch Details:")
print(f"ID: {details['id']}")
print(f"Backend: {details['backend_name']}")
print(f"State: {details['state']}")
print(f"Max Time: {details['max_time']} seconds")
print(f"Accepting Jobs: {details['accepting_jobs']}")
print(f"Mode: {details['mode']}")

### Handling Large Batches of Circuits

<span style="color:red"> **Running the cell below will use around 1 minute from your IBM account credit, you can decrease the number of circuits to decrease circuit run time.** </span>


In [ ]:
# Create a large number of circuits
max_circuits_per_batch = 10 # Limit per batch submission
total_circ = 50
circuits = [random_circuits(3, 3) for _ in range(total_circ)]

# Partition circuits into manageable batches
all_partitioned_circuits = []
for i in range(0, len(circuits), max_circuits_per_batch):
  all_partitioned_circuits.append(circuits[i : i + max_circuits_per_batch])

jobs = []


# Submit partitioned batches
with Batch(backend=backend):
  sampler = Sampler()
  for i, partitioned_circuits in enumerate(all_partitioned_circuits):
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_circuit = pm.run(partitioned_circuits)
    job = sampler.run(isa_circuit)
    jobs.append(job)
    print(f"submitted batch {i+1} with {len(partitioned_circuits)} circuits")


print(total_circuits)
print(f"Partitioned into {len(all_partitioned_circuits)} batches of maximum {max_circuits_per_batch} circuits each")
print(f"Successfully submitted {len(jobs)} jobs to batch")
